# `infer_prompt_matching.ipynb` — 单一 prompt 模式 (soft) 的推理 + 可视化

这个 notebook 是 **配置驱动** 的推理流水线：

1. **Cell 2 是唯一的旋钮区** — 所有 puzzle / 评测 / VGGT 相关的超参数都集中在这里改。
2. 默认配置如下（你可以随时改）：
   - puzzle bbox **占源图 70%~95%** （`PUZZLE_BOX_SCALE_RANGE = (0.70, 0.95)`），尽量"看到大范围"；
   - 跨 view **overlap 只有 10%~40%**（`PUZZLE_OVERLAP_RANGE = (0.10, 0.40)`），让多视角差异更明显；
   - prompt 编码方式固定为 **`soft`**（`PROMPT_POOL = "soft"`），即把 prompt mask 在每个 patch 内的覆盖率作为 query 加权系数。
3. 流程：
   - 用 `SA1BPuzzlesDataset` 生成 `N_SAMPLES` 个 scene，**每个 scene 的 N 张 puzzle crop 落到磁盘**（`infer_prompt_matching_data/scene_xxx/images/`）。
   - 对每个 scene 跑一次 (Fast)VGGT，提取 L2-normalized patch tokens。
   - 对每条 (scene, instance)：
     1. 用 `encode_mask_prompt_soft` 把 prompt mask → patch grid 上的连续权重；
     2. `compute_similarity` 在所有 view 上做余弦相似度；
     3. `binarize` 做百分位阈值 + 绝对阈值；
     4. 上采样回 224x224 与 GT mask 比较，计算 IoU / AP。
4. 输出（`infer_prompt_matching_outputs/`）：
   - `per_instance_records.csv` — 每条样本的 self-match / IoU / AP；
   - `summary.csv`              — 总体均值 / 中位数；
   - `qual_scene*_inst*.png`    — 可视化（heatmap + 二值 mask + GT 轮廓）。

> 这是 `compare_prompt_token.ipynb` 的"实战"版本：那本笔记是把 6 种 prompt encoding 对比，
> 这本笔记是把胜出的 `soft` 单独跑通，并提供清晰的旋钮让你扫不同的 puzzle 难度。


In [ ]:
"""Cell 1 — imports / 路径 / 设备。

只 import 一次模型；不再重复加载 (Fast)VGGT。
"""
from __future__ import annotations

import shutil
import sys
import time
from dataclasses import dataclass
from pathlib import Path
from typing import Any, Dict, List, Tuple

import cv2
import matplotlib.pyplot as plt
import numpy as np
import torch

REPO_ROOT = Path("/mnt/dataset/marko/code/puzzles").resolve()
TFVM_ROOT = REPO_ROOT / "TrainingFreeVGGTMatching"
SRC_ROOT  = REPO_ROOT / "src"
for p in (TFVM_ROOT, SRC_ROOT, REPO_ROOT):
    sp = str(p)
    if sp not in sys.path:
        sys.path.insert(0, sp)

from prompt_matching.extractor import extract_patch_tokens
from prompt_matching.prompt import (
    encode_mask_prompt,
    encode_mask_prompt_soft,
    encode_mask_prompt_subpatch,
)
from prompt_matching.matcher import (
    binarize,
    compute_similarity,
    compute_similarity_subpatch,
    upsample_patch_map,
)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("REPO_ROOT :", REPO_ROOT)
print("TFVM_ROOT :", TFVM_ROOT)
print("DEVICE    :", DEVICE)


In [ ]:
"""Cell 2 — **唯一的旋钮区**。

所有可调超参数集中在这里：
  * Puzzles 数据生成（`PUZZLE_*`, `SA1B_*`）
  * 收集时的"宁缺毋滥"过滤（`MIN_*`）
  * VGGT / 评测（`CKPT_PATH`, `LAYERS`, `THRESHOLD`, ...）
  * Prompt encoding（这里默认固定 `PROMPT_POOL = "soft"`）

>>> 起手默认配置：bbox 占源图 70~95%（看大），overlap 仅 10~40%（更难、更"多视角"）。<<<
"""

# ============================================================
# (A) puzzle 数据集
# ============================================================
SA1B_ROOT     = "/mnt/dataset/marko/data/sa1b/data"
SHARD_NAMES   = ["sa_000000"]
N_VIEWS       = 6
RESOLUTION    = 224
N_SAMPLES     = 12          # 想跑多少个 scene
MAX_PROMPTS   = 4           # 每个 scene 最多 prompt 数
SEED          = 1234

# === 让 puzzle 子图占大范围、跨 view 重叠较少 ===
PUZZLE_BOX_SCALE_RANGE = (0.70, 0.95)   # bbox 占源图 70~95%（视野"大"）
PUZZLE_OVERLAP_RANGE   = (0.10, 0.40)   # 跨 view overlap 10~40%（视角差更明显）
SA1B_MIN_AREA          = 4096.0         # SA-1B annotation 最小面积（默认 1024 太小）
SA1B_MIN_PREDICTED_IOU = 0.90
SA1B_MIN_STAB_SCORE    = 0.92
MIN_VISIBLE_INSTANCES  = 3              # annotation 至少在多少个 view 里可见

# ============================================================
# (B) 收集 instance 时的过滤
# ============================================================
MIN_PROMPT_PIX  = 64                    # 绝对像素下限
MIN_MASK_FRAC   = 0.05                  # prompt mask 至少占 224*224 的 5%
MIN_GT_FRAC     = 0.04                  # 至少有一个 view 的 GT mask 占图 >= 4%
MIN_VISIBLE_GT  = 3                     # GT 在至少 3 个 view 里前景 > MIN_PROMPT_PIX
MAX_INSTANCES_TOTAL = 24                # 上限：总共最多评测多少条 (scene, instance)

# ============================================================
# (C) VGGT / 评测
# ============================================================
CKPT_PATH       = "/mnt/dataset/marko/code/Depth/vggt/checkpoints/vggt.pt"
LAYERS          = (17, 23)        # 与 scripts.md 默认一致
PROMPT_POOL     = "soft"          # <<< 固定 soft 模式 >>>
QUERY_MODE      = "mean"          # 'mean' / 'max'
THRESHOLD       = "percentile"    # 'percentile' / 'otsu'
PERCENTILE      = 90.0
ABS_THRESHOLD   = 0.5             # 跨 view 假阳防护
PER_VIEW_THRESH = True

# ============================================================
# (D) 输出目录
# ============================================================
EVAL_DATA_ROOT = TFVM_ROOT / "infer_prompt_matching_data"
EVAL_OUT_ROOT  = TFVM_ROOT / "infer_prompt_matching_outputs"
EVAL_DATA_ROOT.mkdir(parents=True, exist_ok=True)
EVAL_OUT_ROOT.mkdir(parents=True, exist_ok=True)

print("=" * 60)
print(f"PROMPT_POOL          = {PROMPT_POOL!r}")
print(f"QUERY_MODE           = {QUERY_MODE!r}")
print(f"THRESHOLD            = {THRESHOLD!r} @ p={PERCENTILE}, abs={ABS_THRESHOLD}")
print(f"LAYERS               = {LAYERS}")
print("-" * 60)
print(f"PUZZLE_BOX_SCALE     = {PUZZLE_BOX_SCALE_RANGE}")
print(f"PUZZLE_OVERLAP       = {PUZZLE_OVERLAP_RANGE}")
print(f"SA1B_MIN_AREA        = {SA1B_MIN_AREA}")
print(f"MIN_VISIBLE_INSTANCES= {MIN_VISIBLE_INSTANCES}")
print(f"MIN_MASK_FRAC        = {MIN_MASK_FRAC}")
print(f"MIN_GT_FRAC          = {MIN_GT_FRAC}")
print(f"MIN_VISIBLE_GT       = {MIN_VISIBLE_GT}")
print(f"N_SAMPLES (scenes)   = {N_SAMPLES}")
print(f"MAX_INSTANCES_TOTAL  = {MAX_INSTANCES_TOTAL}")
print("=" * 60)


In [ ]:
"""Cell 3 — 用 SA-1B Puzzles 数据集生成评测样本，并把每个 scene 的 N 张 puzzle crop 写到磁盘。

VGGT 的 extractor 是按目录读图的（沿用 dataloader 预处理），所以这里把 puzzle crop
落盘到 `EVAL_DATA_ROOT/scene_xxx/images/view_vv.png`，再把 prompt mask + GT mask 留在内存里。
"""
import types
from sa1b.sa1b_puzzles_dataset import SA1BPuzzlesDataset
from sa1b.puzzles_bridge import generate_puzzle_crops as _gen_puzzle_crops

# 清空上一次的 scene 缓存，避免新旧子图混在一起
for old_scene in EVAL_DATA_ROOT.glob("scene_*"):
    if old_scene.is_dir():
        shutil.rmtree(old_scene)

ds = SA1BPuzzlesDataset(
    sa1b_root=SA1B_ROOT,
    shard_names=SHARD_NAMES,
    n_views=N_VIEWS,
    resolution=RESOLUTION,
    max_instances=16,
    max_prompts=MAX_PROMPTS,
    min_box_size_scale=PUZZLE_BOX_SCALE_RANGE[0],
    max_box_size_scale=PUZZLE_BOX_SCALE_RANGE[1],
    min_overlap=PUZZLE_OVERLAP_RANGE[0],
    max_overlap=PUZZLE_OVERLAP_RANGE[1],
    min_area=SA1B_MIN_AREA,
    min_predicted_iou=SA1B_MIN_PREDICTED_IOU,
    min_stability_score=SA1B_MIN_STAB_SCORE,
    min_visible_instances=MIN_VISIBLE_INSTANCES,
    seed=SEED,
    augment=False,
)
print(f"SA1BPuzzlesDataset: {len(ds)} samples in shards={SHARD_NAMES}")

# Monkey-patch _load_sample 让它在正常返回之外，顺便把
#   * pr.image  — 源 SA-1B 图 (H_src, W_src, 3) uint8
#   * pr.boxes  — N 个 puzzle bbox，xyxy 坐标落在源图上
# 挂到实例属性上，供 collection 循环取用。seed 决定性，生成两次也不会不一致。
_orig_load_sample = SA1BPuzzlesDataset._load_sample

def _load_sample_with_source(self, idx):
    shard, stem = self._index[idx]
    sample_seed = self.seed + idx
    sample = self.reader.load_sample(shard, stem)
    pr = _gen_puzzle_crops(sample, seed=sample_seed, **self.puzzle_kwargs)
    self._last_source_image = pr.image.copy()             # (H_src, W_src, 3) uint8 RGB
    self._last_source_boxes = [tuple(int(x) for x in b) for b in pr.boxes]
    return _orig_load_sample(self, idx)

ds._load_sample = types.MethodType(_load_sample_with_source, ds)


def _save_views_to_disk(scene_dir: Path, imgs_n3hw: torch.Tensor) -> List[Path]:
    """imgs_n3hw: (N, 3, H, W) float in [0,1] -> N 张 PNG 写到 scene_dir/images/。"""
    imgs_dir = scene_dir / "images"
    if imgs_dir.exists():
        shutil.rmtree(imgs_dir)
    imgs_dir.mkdir(parents=True, exist_ok=True)
    paths: List[Path] = []
    for v in range(imgs_n3hw.shape[0]):
        rgb = (imgs_n3hw[v].permute(1, 2, 0).clamp(0, 1).numpy() * 255).astype(np.uint8)
        bgr = cv2.cvtColor(rgb, cv2.COLOR_RGB2BGR)
        p = imgs_dir / f"view_{v:02d}.png"
        cv2.imwrite(str(p), bgr)
        paths.append(p)
    return paths


@dataclass
class EvalInstance:
    scene_id: int
    inst_id: int
    image_paths: List[Path]
    prompt_view: int                  # which view the prompt is on
    prompt_mask_orig: np.ndarray      # (R, R) bool — input to encode_*
    gt_masks_orig: np.ndarray         # (N, R, R) bool — for IoU/AP eval
    prompt_frac: float
    gt_max_frac: float
    visible_views: int
    source_image: np.ndarray          # (H_src, W_src, 3) uint8 RGB — 原始 SA-1B 图
    source_boxes: List[Tuple[int, int, int, int]]  # N 个 (x1, y1, x2, y2) in source px


# 为了在 instance 收集阶段就能"按 patch 数"过滤掉过小的 prompt，
# 我们用 VGGT 'crop' 模式后的 patch grid (37, 37) 做一次模拟编码。
_FAKE_META_FOR_FILTER = dict(
    orig_H=RESOLUTION, orig_W=RESOLUTION,
    H=518, W=518, H_p=37, W_p=37, patch_size=14,
)
MIN_HARD_PATCH = 6   # soft 模式不强求 hard 编码非空，但太小的 prompt 还是无意义


eval_instances: List[EvalInstance] = []
seen_scenes = 0
attempts   = 0
TOTAL_TRIES = N_SAMPLES * 8

while seen_scenes < N_SAMPLES and attempts < TOTAL_TRIES:
    attempts += 1
    sample = ds[attempts]
    imgs   = sample["imgs"]
    pms    = sample["prompt_masks"]
    gtm    = sample["gt_consistent_masks"]
    pviews = sample["prompt_views"]
    if pms.sum() < MIN_PROMPT_PIX or gtm.sum() < MIN_PROMPT_PIX:
        continue
    # 从 monkey-patch 处拿源图 + boxes（按 seed 决定性生成的 puzzle 结果）
    src_img   = getattr(ds, "_last_source_image", None)
    src_boxes = getattr(ds, "_last_source_boxes", None)
    if src_img is None or src_boxes is None:
        continue
    seen_scenes += 1

    scene_dir = EVAL_DATA_ROOT / f"scene_{seen_scenes:03d}"
    image_paths = _save_views_to_disk(scene_dir, imgs)
    # 也顺便把源图写一份到 scene_dir 方便肉眼查看
    cv2.imwrite(str(scene_dir / "source.png"), cv2.cvtColor(src_img, cv2.COLOR_RGB2BGR))

    M = pms.shape[1]
    R = imgs.shape[-1]
    R2 = float(R * R)
    for m in range(M):
        pv = int(pviews[m].item())
        prompt_m = (pms[pv, m].numpy() > 0.5)
        gt_n     = (gtm[:, m].numpy() > 0.5)

        prompt_frac   = float(prompt_m.sum() / R2)
        per_view_pix  = gt_n.reshape(N_VIEWS, -1).sum(axis=1)
        gt_max_frac   = float(per_view_pix.max() / R2)
        visible_views = int((per_view_pix > MIN_PROMPT_PIX).sum())

        if prompt_m.sum() < MIN_PROMPT_PIX:        continue
        if gt_n.sum()    < MIN_PROMPT_PIX:         continue
        if prompt_frac   < MIN_MASK_FRAC:          continue
        if gt_max_frac   < MIN_GT_FRAC:            continue
        if visible_views < MIN_VISIBLE_GT:         continue

        try:
            patch_mask_hard = encode_mask_prompt(
                prompt_m, _FAKE_META_FOR_FILTER, coverage_threshold=0.3,
            )
            K_hard = int(patch_mask_hard.sum())
        except Exception:
            K_hard = 0
        if K_hard < MIN_HARD_PATCH:
            continue

        eval_instances.append(EvalInstance(
            scene_id        = seen_scenes,
            inst_id         = m,
            image_paths     = image_paths,
            prompt_view     = pv,
            prompt_mask_orig= prompt_m,
            gt_masks_orig   = gt_n,
            prompt_frac     = prompt_frac,
            gt_max_frac     = gt_max_frac,
            visible_views   = visible_views,
            source_image    = src_img,
            source_boxes    = list(src_boxes),
        ))

eval_instances = eval_instances[:MAX_INSTANCES_TOTAL]
print()
print(f"Collected {len(eval_instances)} eval instances "
      f"across {len({e.scene_id for e in eval_instances})} scenes "
      f"(attempts={attempts}, scenes_seen={seen_scenes}).")
if eval_instances:
    fr = np.array([e.prompt_frac for e in eval_instances])
    gf = np.array([e.gt_max_frac for e in eval_instances])
    vv = np.array([e.visible_views for e in eval_instances])
    print(f"  prompt_frac  : min={fr.min():.3f} mean={fr.mean():.3f} max={fr.max():.3f}")
    print(f"  gt_max_frac  : min={gf.min():.3f} mean={gf.mean():.3f} max={gf.max():.3f}")
    print(f"  visible_views: min={int(vv.min())} mean={vv.mean():.1f} max={int(vv.max())}")

# 第一条 instance 的 sanity-check 可视化：
#   左：源图 + 所有 N 个 puzzle bbox；prompt view 红色粗框。
#   右四格：prompt view RGB、prompt mask、GT @ prompt view、另一 view RGB + GT 叠加。
if eval_instances:
    from matplotlib import patches as _mpatches
    from matplotlib import cm as _mpl_cm

    e0 = eval_instances[0]
    cmap_ = _mpl_cm.get_cmap("tab10")

    fig = plt.figure(figsize=(18, 4.2))
    gs  = fig.add_gridspec(1, 5, width_ratios=[1.4, 1, 1, 1, 1])

    # --- 源图 + bboxes ---
    ax_src = fig.add_subplot(gs[0, 0])
    src = e0.source_image
    H_src, W_src = src.shape[:2]
    ax_src.imshow(src)
    for v, (x1, y1, x2, y2) in enumerate(e0.source_boxes):
        is_ref = (v == e0.prompt_view)
        color  = "red" if is_ref else cmap_(v % 10)[:3]
        lw     = 3.5 if is_ref else 1.6
        rect = _mpatches.Rectangle((x1, y1), x2 - x1, y2 - y1,
                                   linewidth=lw, edgecolor=color, facecolor="none")
        ax_src.add_patch(rect)
        ax_src.text(x1 + 4, y1 + 18, f"v{v}" + (" (ref)" if is_ref else ""),
                    fontsize=10, color="white",
                    bbox=dict(boxstyle="round,pad=0.2", fc=color, ec="none", alpha=0.85))
    ax_src.set_xlim(0, W_src); ax_src.set_ylim(H_src, 0)
    ax_src.set_xticks([]); ax_src.set_yticks([])
    ax_src.set_title(f"source {W_src}x{H_src}  N={len(e0.source_boxes)} bboxes")

    # --- 四格：prompt view / prompt mask / GT @ prompt view / GT @ another view ---
    bgr0 = cv2.imread(str(e0.image_paths[e0.prompt_view]))
    a = fig.add_subplot(gs[0, 1]); a.imshow(cv2.cvtColor(bgr0, cv2.COLOR_BGR2RGB))
    a.set_title(f"v{e0.prompt_view} (prompt)"); a.axis("off")

    a = fig.add_subplot(gs[0, 2]); a.imshow(e0.prompt_mask_orig, cmap="magma")
    a.set_title(f"prompt mask  frac={e0.prompt_frac:.2f}"); a.axis("off")

    a = fig.add_subplot(gs[0, 3]); a.imshow(e0.gt_masks_orig[e0.prompt_view], cmap="viridis")
    a.set_title("GT @ prompt view"); a.axis("off")

    other = (e0.prompt_view + 1) % len(e0.image_paths)
    bgr1 = cv2.imread(str(e0.image_paths[other]))
    a = fig.add_subplot(gs[0, 4])
    a.imshow(cv2.cvtColor(bgr1, cv2.COLOR_BGR2RGB))
    a.imshow(e0.gt_masks_orig[other], cmap="viridis", alpha=0.45)
    a.set_title(f"GT @ v{other}  visible_views={e0.visible_views}"); a.axis("off")

    plt.tight_layout()
    plt.show()


In [ ]:
"""Cell 4 — 一次性提取每个 scene 的 patch tokens 并缓存。

每个 scene 的 N 张 puzzle crop 共享 image_dir，所以同一 scene 只跑一次 VGGT。
`token_cache[scene_id] = (tokens, meta)`。
"""
token_cache: Dict[int, Tuple[torch.Tensor, Dict]] = {}

scene_to_paths: Dict[int, List[Path]] = {}
for e in eval_instances:
    scene_to_paths.setdefault(e.scene_id, e.image_paths)

t0 = time.time()
for sid, paths in scene_to_paths.items():
    image_dir = str(Path(paths[0]).parent)
    print(f"\n[scene {sid}] extracting patch tokens (N={len(paths)})...")
    tokens, meta = extract_patch_tokens(
        image_dir=image_dir,
        ckpt_path=CKPT_PATH,
        layers=LAYERS,
        device=DEVICE,
    )
    token_cache[sid] = (tokens, meta)
print(f"\n[cache] populated {len(token_cache)} scenes in {time.time()-t0:.1f}s")


In [ ]:
"""Cell 5 — 推理核心循环：用 PROMPT_POOL='soft' 跑每条 (scene, instance)。

对每条 instance：
  1. encode_mask_prompt_soft(prompt_mask, meta)     -> (H_p, W_p) 连续权重 in [0, 1]
  2. compute_similarity(tokens, ref, patch_mask)    -> (N, H_p, W_p) 余弦相似度
  3. binarize(sim, percentile + abs_threshold)      -> (N, H_p, W_p) 二值
  4. upsample_patch_map(...)                        -> (N, R, R)
  5. 对其他 view 算 IoU / AP，对 ref view 算 self-match。

为了支持 PROMPT_POOL 也能切到 'hard' / 'subpatch' 试一下而不重写代码，这里写了通用分支，
但默认走 'soft'。
"""
import warnings


def _average_precision(scores: np.ndarray, labels: np.ndarray) -> float:
    s = scores.reshape(-1).astype(np.float64)
    y = labels.reshape(-1).astype(bool)
    P = int(y.sum())
    if P == 0:
        return float("nan")
    order = np.argsort(-s, kind="mergesort")
    y_sorted = y[order]
    tp = np.cumsum(y_sorted)
    fp = np.cumsum(~y_sorted)
    precision = tp / np.maximum(tp + fp, 1)
    recall    = tp / P
    rec_prev = np.concatenate(([0.0], recall[:-1]))
    return float(np.sum(precision * (recall - rec_prev)))


def _iou(a_bool: np.ndarray, b_bool: np.ndarray) -> float:
    a = a_bool.astype(bool); b = b_bool.astype(bool)
    inter = float((a & b).sum())
    union = float((a | b).sum())
    return inter / union if union > 0 else float("nan")


def _infer_one(inst: EvalInstance, tokens: torch.Tensor, meta: Dict) -> Dict[str, Any]:
    if PROMPT_POOL == "soft":
        patch_mask = encode_mask_prompt_soft(inst.prompt_mask_orig, meta)
        K = int((patch_mask > 1e-3).sum())
        if patch_mask.sum() <= 0:
            raise RuntimeError("soft prompt sums to 0")
        sim = compute_similarity(
            tokens, ref_view_idx=inst.prompt_view,
            prompt_patch_mask=patch_mask, query_mode=QUERY_MODE,
        )
        coverage_for_self = patch_mask
    elif PROMPT_POOL == "hard":
        patch_mask = encode_mask_prompt(inst.prompt_mask_orig, meta, coverage_threshold=0.3)
        K = int(patch_mask.sum())
        if K == 0:
            raise RuntimeError("hard prompt has 0 patches")
        sim = compute_similarity(
            tokens, ref_view_idx=inst.prompt_view,
            prompt_patch_mask=patch_mask, query_mode=QUERY_MODE,
        )
        coverage_for_self = patch_mask.astype(np.float32)
    elif PROMPT_POOL == "subpatch":
        coords, weights, coverage = encode_mask_prompt_subpatch(
            inst.prompt_mask_orig, meta, mode="dense_gauss",
            stride_px=2, gaussian_sigma_frac=0.4,
        )
        K = int(coords.shape[0])
        if K == 0:
            raise RuntimeError("subpatch produced 0 samples")
        sim = compute_similarity_subpatch(
            tokens, ref_view_idx=inst.prompt_view,
            sample_coords=coords, sample_weights=weights,
            query_mode=QUERY_MODE,
        )
        coverage_for_self = coverage
    else:
        raise ValueError(f"Unknown PROMPT_POOL={PROMPT_POOL!r}")

    sim_np = sim.detach().float().cpu().numpy()                 # (N, H_p, W_p)
    binm = binarize(
        sim, method=THRESHOLD, percentile=PERCENTILE,
        per_view=PER_VIEW_THRESH, abs_threshold=ABS_THRESHOLD,
    ).cpu().numpy().astype(bool)

    N = sim_np.shape[0]
    R = inst.gt_masks_orig.shape[1]
    sim_hi  = np.stack([upsample_patch_map(sim_np[v], R, R, "bilinear") for v in range(N)], axis=0)
    mask_hi = np.stack([upsample_patch_map(binm[v].astype(np.float32), R, R, "nearest").astype(bool) for v in range(N)], axis=0)

    gt = inst.gt_masks_orig
    other_views = [v for v in range(N) if v != inst.prompt_view]

    iou_per = [_iou(mask_hi[v], gt[v]) for v in other_views]
    ap_per  = []
    for v in other_views:
        with warnings.catch_warnings():
            warnings.simplefilter("ignore")
            ap_per.append(_average_precision(sim_hi[v], gt[v]))

    w = coverage_for_self.astype(np.float32)
    self_match = float((sim_np[inst.prompt_view] * w).sum() / (w.sum() + 1e-6))

    return {
        "K": K,
        "sim_hi":  sim_hi,
        "mask_hi": mask_hi,
        "self_match_ref": self_match,
        "iou_other": float(np.nanmean(iou_per)) if iou_per else float("nan"),
        "ap_other":  float(np.nanmean(ap_per))  if ap_per  else float("nan"),
        "iou_per_view": iou_per,
        "ap_per_view":  ap_per,
    }


records: List[Dict[str, Any]] = []
viz_blob: Dict[Tuple[int, int], Dict[str, Any]] = {}

t0 = time.time()
for ei, inst in enumerate(eval_instances):
    tokens, meta = token_cache[inst.scene_id]
    try:
        res = _infer_one(inst, tokens, meta)
    except Exception as exc:
        print(f"[{ei+1}/{len(eval_instances)}] scene={inst.scene_id} inst={inst.inst_id}  FAILED ({exc})")
        records.append({
            "scene": inst.scene_id, "inst": inst.inst_id,
            "prompt_view": inst.prompt_view,
            "prompt_frac": inst.prompt_frac, "gt_max_frac": inst.gt_max_frac,
            "visible_views": inst.visible_views,
            "K": np.nan, "self_match_ref": np.nan,
            "iou_other": np.nan, "ap_other": np.nan,
            "error": str(exc),
        })
        continue
    records.append({
        "scene": inst.scene_id, "inst": inst.inst_id,
        "prompt_view": inst.prompt_view,
        "prompt_frac": inst.prompt_frac, "gt_max_frac": inst.gt_max_frac,
        "visible_views": inst.visible_views,
        "K": res["K"],
        "self_match_ref": res["self_match_ref"],
        "iou_other": res["iou_other"],
        "ap_other":  res["ap_other"],
    })
    viz_blob[(inst.scene_id, inst.inst_id)] = res
    print(f"[{ei+1}/{len(eval_instances)}] scene={inst.scene_id} inst={inst.inst_id} pv={inst.prompt_view}  "
          f"K={res['K']:4d}  self={res['self_match_ref']:.3f}  "
          f"IoU={res['iou_other']:.3f}  AP={res['ap_other']:.3f}")
print(f"\n[infer] all done in {time.time()-t0:.1f}s")


In [ ]:
"""Cell 6 — 汇总到 DataFrame，打印总体均值/中位数，并落盘 CSV。"""
import pandas as pd

df = pd.DataFrame.from_records(records)
print("rows:", len(df))
df_ok = df.dropna(subset=["iou_other", "ap_other"])

if len(df_ok):
    summary = pd.DataFrame({
        "n":     [len(df_ok)],
        "K_mean":               [df_ok["K"].mean()],
        "self_match_mean":      [df_ok["self_match_ref"].mean()],
        "self_match_median":    [df_ok["self_match_ref"].median()],
        "iou_other_mean":       [df_ok["iou_other"].mean()],
        "iou_other_median":     [df_ok["iou_other"].median()],
        "ap_other_mean":        [df_ok["ap_other"].mean()],
        "ap_other_median":      [df_ok["ap_other"].median()],
        "prompt_frac_mean":     [df_ok["prompt_frac"].mean()],
        "gt_max_frac_mean":     [df_ok["gt_max_frac"].mean()],
        "visible_views_mean":   [df_ok["visible_views"].mean()],
    })
    print("\n[summary]")
    print(summary.to_string(index=False, float_format=lambda x: f"{x:.4f}"))
else:
    summary = pd.DataFrame()
    print("[summary] no valid rows")

df.to_csv(EVAL_OUT_ROOT / "per_instance_records.csv", index=False)
summary.to_csv(EVAL_OUT_ROOT / "summary.csv", index=False)
print(f"\nWrote {EVAL_OUT_ROOT/'per_instance_records.csv'}")
print(f"Wrote {EVAL_OUT_ROOT/'summary.csv'}")

df.head()


In [ ]:
"""Cell 7 — 把 IoU/AP 的分布画出来，便于一眼看 "好/差" 的样本占比。"""

if len(df_ok):
    fig, axes = plt.subplots(1, 3, figsize=(13, 3.6))
    for ax, col, title, color in zip(
        axes,
        ["self_match_ref", "iou_other", "ap_other"],
        ["self-match @ ref view", "mean IoU on other views", "mean AP on other views"],
        ["#1976d2", "#43a047", "#fb8c00"],
    ):
        vals = df_ok[col].dropna().values
        ax.hist(vals, bins=12, color=color, alpha=0.85, edgecolor="white")
        ax.axvline(vals.mean(),   color="k", lw=1.4, ls="--", label=f"mean  {vals.mean():.3f}")
        ax.axvline(np.median(vals),color="r",lw=1.4, ls=":",  label=f"med   {np.median(vals):.3f}")
        ax.set_title(title); ax.legend(fontsize=8)
        ax.grid(axis="y", linestyle=":", alpha=0.4)
    fig.suptitle(f"PROMPT_POOL='{PROMPT_POOL}'  |  bbox={PUZZLE_BOX_SCALE_RANGE}  overlap={PUZZLE_OVERLAP_RANGE}",
                 fontsize=11)
    plt.tight_layout()
    out_path = EVAL_OUT_ROOT / "metric_distributions.png"
    plt.savefig(out_path, dpi=120)
    plt.show()
    print(f"Wrote {out_path}")
else:
    print("[hist] nothing to plot — df_ok is empty.")


In [ ]:
"""Cell 8 — 定性可视化。

每条 instance 画两张图：
  (A) `qual_scene..._source.png`
      原始 SA-1B 源图 + N 个 puzzle bbox（按 view 上色）。
      prompt view 的 bbox 用 **红色粗框**，其他用细边。
      bbox 左上角标 `v{idx}`。
  (B) `qual_scene..._pred.png`
      2×N 网格：
        row 0 = RGB + heatmap overlay；
        row 1 = RGB + GT (蓝虚线) + pred (lime)；
      ref view 用红色坐标轴 + 黄色 prompt 轮廓区分。
"""
from matplotlib import patches as mpatches
from matplotlib import cm as mpl_cm


def _draw_contour(ax, mask: np.ndarray, color: str, lw: float = 1.6, ls: str = "-"):
    if mask is None or mask.sum() == 0:
        return
    ax.contour(mask.astype(float), levels=[0.5], colors=color, linewidths=lw, linestyles=ls)


def _view_palette(n: int) -> List[Tuple[float, float, float]]:
    """从 tab10 里挑 N 个饱和度高的颜色（用于每个 view 的 bbox）。"""
    cmap = mpl_cm.get_cmap("tab10")
    return [cmap(i % 10)[:3] for i in range(n)]


def visualize_source(scene_id: int, inst_id: int):
    inst = next(e for e in eval_instances if e.scene_id == scene_id and e.inst_id == inst_id)
    src  = inst.source_image                          # (H_src, W_src, 3) uint8 RGB
    boxes = inst.source_boxes                         # list[(x1,y1,x2,y2)]
    H_src, W_src = src.shape[:2]
    N = len(boxes)
    palette = _view_palette(N)

    fig, ax = plt.subplots(1, 1, figsize=(8, 8))
    ax.imshow(src)
    for v, ((x1, y1, x2, y2), color) in enumerate(zip(boxes, palette)):
        is_ref = (v == inst.prompt_view)
        lw   = 4.0 if is_ref else 2.0
        ec   = "red" if is_ref else color
        rect = mpatches.Rectangle(
            (x1, y1), x2 - x1, y2 - y1,
            linewidth=lw, edgecolor=ec, facecolor="none",
        )
        ax.add_patch(rect)
        # 背景板 + label（避免白字看不清）
        tag = f"v{v}" + (" (ref)" if is_ref else "")
        ax.text(
            x1 + 4, y1 + 22, tag,
            fontsize=12, color="white", weight="bold",
            bbox=dict(boxstyle="round,pad=0.25", fc=ec, ec="none", alpha=0.85),
        )
    ax.set_xlim(0, W_src)
    ax.set_ylim(H_src, 0)
    ax.set_xticks([]); ax.set_yticks([])
    ax.set_title(
        f"scene={scene_id}  inst={inst_id}  source={W_src}x{H_src}px  N_views={N}  "
        f"(red = prompt view v{inst.prompt_view})",
        fontsize=11,
    )
    plt.tight_layout()
    out_path = EVAL_OUT_ROOT / f"qual_scene{scene_id:03d}_inst{inst_id:02d}_source.png"
    plt.savefig(out_path, dpi=110)
    plt.show()
    print(f"Wrote {out_path}")


def visualize_instance(scene_id: int, inst_id: int):
    inst = next(e for e in eval_instances if e.scene_id == scene_id and e.inst_id == inst_id)
    res  = viz_blob.get((scene_id, inst_id))
    if res is None:
        print(f"[viz] (scene={scene_id}, inst={inst_id}) failed during inference; skip.")
        return

    # === (A) 先画源图 + bboxes ===
    visualize_source(scene_id, inst_id)

    # === (B) 再画 N 个 view 的 heatmap / pred vs GT ===
    rgb_views = [cv2.cvtColor(cv2.imread(str(p)), cv2.COLOR_BGR2RGB) for p in inst.image_paths]
    N = len(rgb_views)
    palette = _view_palette(N)

    fig, axes = plt.subplots(2, N, figsize=(2.2 * N, 5.0))

    for c in range(N):
        ax = axes[0, c]
        ax.imshow(rgb_views[c])
        heat = res["sim_hi"][c]
        lo, hi = float(np.percentile(heat, 5)), float(heat.max())
        lo = max(lo, 0.0); hi = max(hi, lo + 1e-3)
        ax.imshow(np.clip((heat - lo) / (hi - lo), 0, 1), cmap="jet", alpha=0.45)
        if c == inst.prompt_view:
            _draw_contour(ax, inst.prompt_mask_orig, color="yellow", lw=1.8)
        # 用和源图 bbox 一致的颜色标记 view（方便前后对照）
        ec = "red" if c == inst.prompt_view else palette[c]
        for spine in ax.spines.values():
            spine.set_edgecolor(ec)
            spine.set_linewidth(2.5 if c == inst.prompt_view else 2.0)
        ax.set_title(f"v{c}{' (ref)' if c == inst.prompt_view else ''}", fontsize=10)
        ax.set_xticks([]); ax.set_yticks([])
    axes[0, 0].set_ylabel("heatmap", fontsize=10)

    for c in range(N):
        ax = axes[1, c]
        ax.imshow(rgb_views[c])
        _draw_contour(ax, inst.gt_masks_orig[c], color="dodgerblue", lw=1.4, ls="--")
        _draw_contour(ax, res["mask_hi"][c],     color="lime",       lw=1.6)
        if c == inst.prompt_view:
            _draw_contour(ax, inst.prompt_mask_orig, color="yellow", lw=1.6)
        ec = "red" if c == inst.prompt_view else palette[c]
        for spine in ax.spines.values():
            spine.set_edgecolor(ec)
            spine.set_linewidth(2.5 if c == inst.prompt_view else 2.0)
        ax.set_xticks([]); ax.set_yticks([])
    axes[1, 0].set_ylabel("pred (lime) vs GT (blue)", fontsize=10)

    fig.suptitle(
        f"scene={scene_id}  inst={inst_id}  pv={inst.prompt_view}   "
        f"PROMPT_POOL='{PROMPT_POOL}'   "
        f"self={res['self_match_ref']:.3f}  IoU={res['iou_other']:.3f}  AP={res['ap_other']:.3f}",
        fontsize=11,
    )
    plt.tight_layout()
    out_path = EVAL_OUT_ROOT / f"qual_scene{scene_id:03d}_inst{inst_id:02d}_pred.png"
    plt.savefig(out_path, dpi=110)
    plt.show()
    print(f"Wrote {out_path}")


# 选择前 N_SHOW 条画图（默认 4 条；可以改大）
N_SHOW = min(4, len(eval_instances))
for e in eval_instances[:N_SHOW]:
    visualize_instance(e.scene_id, e.inst_id)


## 旋钮速查表（都在 Cell 2）

### Puzzle 难度
| 旋钮 | 作用 | 当前值 | 想"更难"就调 |
|---|---|---|---|
| `PUZZLE_BOX_SCALE_RANGE` | 每张子图占源图的比例 | `(0.70, 0.95)` | 调小 → 子图更小、更"局部" |
| `PUZZLE_OVERLAP_RANGE`   | 跨 view 共享面积比例 | `(0.10, 0.40)` | 调小 → view 之间几乎不重叠 |
| `N_VIEWS`                | 一个 scene 的 view 数 | `6` | 调大 → 评测更稳定 |
| `RESOLUTION`             | 单张 puzzle crop 分辨率 | `224` | (Fast)VGGT 的输入 patch grid 是 (518/14)² = 37² |

### SA-1B 选 mask
| 旋钮 | 作用 | 当前值 |
|---|---|---|
| `SA1B_MIN_AREA` | 丢掉太小的标注 | `4096` 像素 |
| `SA1B_MIN_PREDICTED_IOU` / `SA1B_MIN_STAB_SCORE` | SAM 自评分阈值 | `0.90 / 0.92` |
| `MIN_VISIBLE_INSTANCES` | annotation 至少在多少 view 里可见 | `3` |

### Instance 收集时的"宁缺毋滥"过滤
| 旋钮 | 作用 | 当前值 |
|---|---|---|
| `MIN_MASK_FRAC` | prompt mask 至少占 224×224 的比例 | `0.05` |
| `MIN_GT_FRAC`   | 至少有一个 view 的 GT 占图比例 | `0.04` |
| `MIN_VISIBLE_GT`| GT 在多少 view 里有前景 | `3` |

### Prompt encoding
| 旋钮 | 作用 | 当前值 |
|---|---|---|
| `PROMPT_POOL`  | `'soft'` / `'hard'` / `'subpatch'` | **`'soft'`** ← 已定 |
| `QUERY_MODE`   | 多 query 聚合方式 | `'mean'` |

### 阈值化 / 上采样
| 旋钮 | 作用 | 当前值 |
|---|---|---|
| `THRESHOLD`       | `'percentile'` / `'otsu'` | `'percentile'` |
| `PERCENTILE`      | percentile 阈值 | `90.0`（取每个 view top-10% sim） |
| `ABS_THRESHOLD`   | 绝对阈值，过滤跨 view 假阳 | `0.5` |
| `PER_VIEW_THRESH` | 是否每个 view 各自取阈值 | `True` |

---

## 输出文件
所有产物落在 `infer_prompt_matching_outputs/`：

* `per_instance_records.csv` — 每条样本的诊断字段 + 指标；
* `summary.csv`              — 整体均值 / 中位数；
* `metric_distributions.png` — 三项核心指标的直方图；
* `qual_scene*_inst*.png`    — 定性可视化（heatmap / pred vs GT）。

## 想做什么
* **想看 puzzle 更难时 soft 还撑不撑得住** → 把 `PUZZLE_OVERLAP_RANGE = (0.05, 0.20)` 再跑一遍。
* **想看大 prompt vs 小 prompt** → 调整 `MIN_MASK_FRAC` / `SA1B_MIN_AREA`。
* **想换模型** → 改 `CKPT_PATH` 即可（`extract_patch_tokens` 自动按 ckpt 选 VGGT / FastVGGT）。
* **想看二值化更激进/保守** → 调 `PERCENTILE`（90 → 95 更尖锐，80 更宽松），或改 `THRESHOLD='otsu'`。
